In [10]:
import os
import uuid
import requests

URL = "https://llm-gate-np.localiza.dev/llm-gate/v2/chat/completions"

headers = {
    "Content-Type": "application/json",
    "X-Correlation-ID": "7370939e-dbaa-4a07-83e8-c1912b750fd1",
# api_key removida
    "api_key": os.getenv("API_KEY")
}

payload = {
    "model": "gpt-4o-mini",
    "messages": [
        {"role": "user", "content": "Olá! Este é um teste de conectividade."}
    ],
}

try:
    resp = requests.post(URL, headers=headers, json=payload)
    print("Status:", resp.status_code)
    print("Body:", resp.text)
    resp.raise_for_status()
except requests.exceptions.RequestException as e:
    print("Erro na requisição:", e)


Status: 200
Body: {"id":"chatcmpl-EGUzJJxwsMBtn9EnL5RJ57eYyLSEW","object":"chat.completion","created":1787601073,"model":"gpt-4o-mini-2024-07-18","choices":[{"index":0,"message":{"role":"assistant","content":"Olá! Está tudo funcionando bem. Como posso ajudá-lo hoje?","annotations":[]},"finish_reason":"stop"}],"usage":{"prompt_tokens":17,"completion_tokens":14,"total_tokens":31,"prompt_tokens_details":{"cached_tokens":0,"audio_tokens":0},"completion_tokens_details":{"reasoning_tokens":0,"audio_tokens":0,"accepted_prediction_tokens":0,"rejected_prediction_tokens":0}},"service_tier":"default"}


In [ ]:
import requests
import json
import math
import openpyxl
import pandas as pd
import uuid
import openpyxl
import re
# import pydata_google_auth
import os
import csv 
from datetime import datetime
from IPython.display import display, HTML
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging

# URL da API
url = "https://llm-gate-np.localiza.dev/llm-gate/v2/chat/completions"
limite_tokens = 10000000

# Cabeçalho da requisição
headers = {
    "Content-Type": "application/json",
    "X-Correlation-ID": "7370939e-dbaa-4a07-83e8-c1912b750fd1",
    "api_key": os.getenv("API_KEY"),
}

# Usuário
user = '133341'

# Caminhos de arquivos
path_base_respondentes = f"C:/Users/{user}/OneDrive - Localiza/Produto e CX - PJ/NPS GESTOR/TABULAÇÃO GENAI/BASE_NPS_GESTOR.xlsx"
sheet_name_base_respondentes = "BASE_RESPONDENTES"
path_comentarios_existentes = f"C:/Users/{user}/OneDrive - Localiza/Produto e CX - PJ/NPS GESTOR/TABULAÇÃO GENAI/teste_python_nps.csv"
path_classificacoes = f"C:/Users/{user}/OneDrive - Localiza/Produto e CX - PJ/NPS GESTOR/TABULAÇÃO GENAI/Classificacoes.xlsx"
path_exemplos = f"C:/Users/{user}/OneDrive - Localiza/Produto e CX - PJ/NPS GESTOR/TABULAÇÃO GENAI/Exemplos.xlsx"
diretorio_log = f"C:/Users/{user}/OneDrive - Localiza/Produto e CX - PJ/NPS GESTOR/TABULAÇÃO GENAI/logs"


In [ ]:
# Carregar a planilha de classificações
workbook_classificacoes = openpyxl.load_workbook(
    f"{path_classificacoes}"
)
sheet_classificacoes = workbook_classificacoes.active

# Inicializar uma lista para armazenar as classificações
classificacoes = []

# Iterar pelas linhas da planilha de classificações
for idx, row in enumerate(sheet_classificacoes.iter_rows(min_row=2, max_col=2, values_only=True), start=1):
    categoria, descricao = row
    classificacoes.append({
        "id": idx,
        "categoria": categoria,
        "descricao": descricao
    })

# Carregar a planilha de exemplos de classificação
workbook_exemplos_classificacao = openpyxl.load_workbook(
    f"{path_exemplos}"
)
sheet_exemplos_classificacao = workbook_exemplos_classificacao.active

# Inicializar uma lista para armazenar os exemplos de classificação
exemplos_classificacao = []

# Iterar pelas linhas da planilha de exemplos de classificação
# for idx, row in enumerate(sheet_exemplos_classificacao.iter_rows(min_row=3, max_col=3, values_only=True), start=1): #Será que min_row=3 mesmo? As vezes a planilha deles começa na segunda linha do excel.
for idx, row in enumerate(sheet_exemplos_classificacao.iter_rows(min_row=2, max_col=3, values_only=True), start=1): 
    Comentario, Exemplo_Classificacao, Motivo_Classificacao = row
    exemplos_classificacao.append({
        "id": idx,
        "Comentario": Comentario,
        "Exemplo_Classificacao": Exemplo_Classificacao,
        "Motivo_Classificacao": Motivo_Classificacao,
    })

# display(sheet_classificacoes)
# display(sheet_exemplos_classificacao)
# print(exemplos_classificacao)

def to_str_safe(x):
    # Trata NaN do pandas/numpy
    if isinstance(x, float) and math.isnan(x):
        return ""
    if x is None:
        return ""
    # Se já é string, tira espaços
    if isinstance(x, str):
        return x.strip()
    # Se for dict/list, serializa para JSON
    if isinstance(x, (dict, list)):
        return json.dumps(x, ensure_ascii=False)
    # Fallback: converte para string
    return str(x).strip()

# Função para criar o corpo da requisição com conteúdo dinâmico
def criar_corpo_requisicao(conteudo_usuario, classificacoes, exemplos_classificacao):
    if not conteudo_usuario:
        # Evita content vazio, que também pode gerar 400
        conteudo_usuario = "Sem comentário disponível."

    classificacoes_str = "Siga as seguintes classificações pré-definidas: "
    for classificacao in classificacoes:
        classificacoes_str += (
            f"{classificacao['id']}. {classificacao['categoria']}: {classificacao['descricao']}"
        )

    exemplos_classificacao_str = "Aqui estão alguns exemplos de classificação: "
    for exemplo in exemplos_classificacao:
        exemplos_classificacao_str += (
            f"{exemplo['id']}. {exemplo['Comentario']}: {exemplo['Exemplo_Classificacao']}, "
            f"Motivo: {exemplo['Motivo_Classificacao']} \n"
        )

    return {
        "model": "gpt-4o-mini",
        "messages": [
                {
                    "role": "system",
                    "content": (
                        "Você está atuando como um classificador de motivos de insatisfação para uma empresa de aluguel de carros."
                        "Analize os comentários dos clientes para atribuir as classificações corretas com base nos motivos que causaram insatisfação."
                        "Siga as instruções:"
                        "1. Leia o comentário cuidadosamente: Identifique os temas de insatisfação expressos pelo cliente em relação aos serviços ou produtos fornecidos pela empresa."
                        "2. No final de todo prompt, você vai classificar os comentários de acordo com as categorias pré-definidas que serão fornecidas até o final do prompt: Associe uma ou mais categorias ao comentário, conforme considerar aplicável."
                    ),
                },
                {
                    "role": "system",
                    "content": (
                        "Algumas regras importantes que você deve saber sobre a empresa antes de classificar os comentários:"
                        "1- Colete todas as informações na sua base de dados sobre a Localiza Rent a Car e sobre o mercado de aluguel de carros para absorver mais contexto no momento de classificar os comentários"
                        "2- Seguem as regras específicas do negócio de aluguel de carros:"
                        "2.1 - A empresa oferece ao cliente duas formas de retirar o carro, uma pelo aplicativo usando a retirada digital e outra de forma convencional com atendimento humano no balcão"
                        "2.2 - Caso o cliente escolha o Fast Retirada Digital para retirar o seu veículo, o ideal é que ele não precise de atendimento humano e faça o processo 100% sozinho. O cliente que não precisa de ajuda é chamado de autônomo ou touchless."
                        "2.3 - A empresa oferece produtos adicionais além do aluguel de carros, como proteção do carro, limpeza garantida caso o cliente entregue o carro sujo demais, condutor adicional ou condutor ilimitado caso o cliente queira que mais de uma pessoa dirija o carro, seguro para o carro e para terceiros e etc"
                        "2.4 - Os carros disponibilizados para os clientes deveriam ser sempre completamente abastecidos e serem entregues limpos."
                        "2.5 - Todo cliente indica um horário de retirar o carro, no caso do Fast Retirada Digital ele pode retirar o veículo uma hora e meia antes do horário agendado e até duas horas depois. O cliente não consegue retirar o carro digitalmente fora dessa janela"
                        "2.6 - Os carros da locadora vão com uma tag de pagamentos colada no parabrisa que permite ao cliente passar por pedágios e cancelas de estacionamento sem precisar parar o veículo, alguns clientes precisam ir até o balcão de atendimento para ativar essa tag"
                        "2.7 - A locadora oferece opção de mais de 20 categorias de carro ao cliente que escolhe a que melhor se adapta as suas necessidades no momento de criação de sua reserva"
                        "2.8 - Todos os clientes pagam um valor de caução, chamado de pré-autorização, no momento da retirada do carro que deve ser devolvido ao cliente no final da locação"
                        "2.9 - A empresa oferece atendimento humano em suas lojas e em caso de acidentes ou dúvidas quando o cliente está fisicamente fora das lojas também oferece atendimento remoto"
                        "2.10 - Alguns clientes mais fiéis participam de um programa de benefícios fidelidade que os fornece algumas vantagens como possibilidade de pegar um carro melhor que o contratado, chamamos isso de upgrade fidelidade"
                        "2.11 - No momento da retirada do carro alugado a agência pode não ter disponível um modelo do grupo selecionado pelo cliente no momento da reserva, nesses casos a agência oferece um carro de categoria superior (upgrade gratúito), mas isso nem sempre é percebido como um benefício pelo cliente."
                        "2.12 - No passado, como forma de agradar os clientes que passavam pelo balcão de atendimento era comum oferecer um upgrade de categoria de carro gratuito. Essa prática foi desestimulada pela empresa, mas alguns clientes citam insatisfação por estarem acostumados a receber um grupos de carros superior ao contratado."
                    ),                   
                },
                            {
                    "role": "system",
                    "content": (
                        "Algumas instruções importantes sobre a sua resposta"
                        "1- Se nenhuma classificação pré-definida se aplicar ao comentário analisado utilize a classificação 'Categoria Não Definida'." 
                        "2- Para comentários absolutamente vagos utilize a categoria 'Comentário Vago' que será fornecido na lista de categorias pré-definidas a seguir."
                        "3- Use categorias genéricas **somente** quando não for possível usar categorias específicas. Por exemplo, utiilize ‘outras cobranças’ para classificar um comentário somente se não for possível usar classificações mais específicas como ‘Cobrança de avaria’ ou 'Cobrança de Hora Extra'." 
                        "4- Utilize somente classificações que indiquem motivo de insatisfação, em caso de haver motivo de insatisfação e satisfação no mesmo comentário, utilize para classificar somente as categorias cujo cliente sinalizou insatisfação."
                        "5- Tente identificar todos os motivos que geraram insatisfação para o cliente relatados no comentário, não somente o motivo que considera principal."
                        "6- No momento de classificar seja bastante objetivo, se atenha ao fatos descritos pelo cliente no comentário no momento de classificar."
                    ),                   
                },
                {
                    "role": "system",
                    "content": (
                        "Vou fornecer a lista de categorias pré-definidas que você deve utilizar na classificação."
                        "Você receberá 2 informações sobre cada categoria. "
                        "Informações em 'Categoria' fornecerão o nome de cada categoria."
                        "Informações em 'Descrição' fornecerão mais descrições de cada categoria."
                        "" + classificacoes_str + "."
                    ),
                },
                {
                    "role": "system",
                    "content": (
                        "Vou fornecer também alguns exemplos de comentários classificados da maneira correta"
                        "1 - Vou fornecer 3 informações sobre os exemplo"
                        "A primeira delas chamada 'Comentario' é uma replica exata do comentario classificado corretamente"
                        "A segunda informação, fornecida em 'Exemplo_Classificacao' exibirá como seria a classificação correta desse comentário"
                        "A terceira informação em 'Motivo_Classificacao', mostrará o motivo da classificação ter sido usada dessa forma."
                        "" + exemplos_classificacao_str + "."
                    ),                   
                },   
                {
                    "role": "system",
                    "content": (
                        """
                        Execute as ações na ordem estabelecida a seguir:
                        1. Relembre todas as categorias pré-definidas.
                        2. Releia os comentários fornecidos como exemplo.
                        3. Antes de classificar, pare e pense: assertividade é MUITO mais importante que velocidade no momento de classificar os comentários.
                        4. Leia o comentário fornecido e identifique todas as possíveis classificações que se aplicam, usando as classificações pré-definidas.
                        5. Pare e pense, verifique se cada classificação identificada foi explicitamente citada pelo cliente no comentário.
                        6. Agora, assuma o papel de um segundo expert em classificação de informações.
                        7. Leia as classificações pré-definidas fornecidas anteriormente.
                        8. Leia os exemplos fornecidos anteriormente para identificar padrões que podem se repetir no comentário atualmente analisado.
                        9. Observe atentamente para garantir que nenhuma categoria genérica foi usada quando uma categoria específica pode ser aplicada.
                        10. Antes de classificar, pare e pense: assertividade é MUITO mais importante que velocidade no momento de classificar os comentários.
                        11. Compare todas as classificações possíveis identificadas anteriormente e escolha as que se aplicam com maior grau de assertividade, justificando sua escolha.
                        12. Pare e pense, agora assuma o papel de um terceiro expert em classificação de informações, selecionado para rever o trabalho de classificação dos outros dois.
                        13. Leia as classificações pré-definidas fornecidas anteriormente.
                        14. Leia os exemplos fornecidos anteriormente.
                        15. Observe atentamente para garantir que a classificação é objetiva e se atém ao que foi citado explicitamente no comentário do cliente, sem imaginar possibilidades que não estão explícitas.
                        16. Revise a classificação escolhida à luz de novas informações ou exemplos fornecidos, e ajuste se necessário.
                        17. Avalie e reporte o grau de certeza (probabilidade estatística de suas classificações estarem certas) para cada classificação escolhida, só use a classificação se houver .
                        18. Me forneça a classificação final no formato JSON especificado abaixo:
                        'Classificação': 'Aqui você deve fornecer as classificações que você entende pertinentes como motivo de **INSATISFAÇÃO DO CLIENTE**"
                        'Motivo da Classificação': 'explicação detalhada do raciocínio utilizado para usar cada classificação pré-definida condensada em uma única coluna, preciso entender sua linha de pensamento e tomada de decisão para te ajudar a melhorar. Me forneça o máximo de informações possíveis sobre porquê entende que as categorias escolhidas se aplicam'."
                        'Dúvida': 'Coloque aqui as dúvidas que teve no momento de classificar o comentário'"
                        'Sugestão de Nova Categoria': 'Adicione aqui sugestões de novas categorias caso você entenda que o cliente esteja insatisfeito ou tenha dúvidas sobre um fator que não se encaixa nas classificações pré-definidas.
                        'Influencia_Exemplos': 'adicione aqui sua percepção de quanto os exemplos que forneci de comentários que você classificou errado no passado influenciaram na classificação atual do comentário, se possível explique o porquê
                        'Percentual de Assertividade': Após realizar a classificaçação aponte estatisticamente a probabilidade de a classificação estar correta.'
                        """
                    ),                   
                },
                {"role": "user", "content": conteudo_usuario},
        ],
        "temperature": 0,
        # "max_tokens": 256,
        "response_format": {"type": "json_object"}
    }


In [ ]:
# Ler o arquivo base
df_NPS_base = pd.read_excel(path_base_respondentes, sheet_name=sheet_name_base_respondentes)

# Inicialize uma lista vazia para armazenar as respostas da API
dados_respostas = []

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(df_NPS_base.head(10))

In [14]:
# Formatar o arquivo base

## Inicializar df_NPS
df_NPS = pd.DataFrame()

## Filtrar por nota NPS específica
df_NPS = df_NPS_base.loc[df_NPS_base["NPS"] <= 8]
# display(df_NPS.head(10))

## Renomear campos com nomes longos
renomeacao_colunas = dict({
    "O que motivou a sua nota?": "Resposta",
    
    "Conta pra gente, o que motivou sua nota acima?": "Motivação Nota Qualidade do Atendimento EV",
    
    "Conta para gente, o que motivou sua nota para o processo de faturamento?": "Motivação Nota Faturamento",
    
    "Caso você já tenha utilizado os serviços de algum parceiro da Localiza (Serviço de Reboque, Oficinas": "Serviços Parceiros",
    
    "Conta para gente, o que motivou sua nota acima sobre a experiência com o prestador de serviço parcei": "Motivação Nota Serviços Parceiros",
    
    "Considerando a redução ou devolução da frota alugada, qual seria o principal motivo? - Selected Choi": "Motivação Sentimento de Compra",
    
    "Considerando a redução ou devolução da frota alugada, qual seria o principal motivo? - Outro(s): – T": "Comentário Sentimento de Compra",
    
    "Qual sua intenção de alugar carro com a Localiza": "Intenção de Alugar Carros",
    
    "Você sente falta de algum tipo de serviço ou facilidade que não é oferecido atualmente pela Localiza": "Sentimento de Falta de Serviços/Facilidade",
})

df_NPS = df_NPS.rename(columns=renomeacao_colunas)


## Adicionar chave única de identificação de respostas
df_NPS["chave"] = ""

for index, row in df_NPS.iterrows():
    
    codigo_cliente = row.get("Código Cliente","")
    email = row.get("E-mail", "")
    data = row.get("Data Pesquisa", "")
    
    if pd.notnull(data) and isinstance(data, pd.Timestamp):
        data_formatada = data.strftime("%d/%m/%Y")
    else:
        data_formatada = str(data)
    
    chave = f"{codigo_cliente} | {email} | {data_formatada}"  
    df_NPS.at[index, "chave"] = chave


## Classificar NPS ainda não classificados
def classificacao_NPS(index, row, notas):
    
    for categoria in notas:
        valor = row.get(categoria, "")
        classificacao = f"Classificação {categoria}"

        if isinstance(valor, str) and valor.strip().lower() == "não aplicável":
            continue
        
        try:
            numero = float(valor)
            if numero <= 6:
                row[classificacao] = "Detrator"
            elif numero <= 8:
                row[classificacao] = "Neutro"
            elif numero <= 10:
                row[classificacao] = "Promotor"
                
        except (ValueError, TypeError):
            continue  # ignora se não for número
    return row

campos_notas_NPS = ["NPS", "Qualidade do atendimento EV", "Frequência de contato EV", "Condições comerciais", "Atendimento 0800V", "WebCorp", "Ceres", "Qualidade do carro", "Faturamento", "ASCLI", "Serviços Parceiros"]

for categoria in campos_notas_NPS:
    nome_campo = f"Classificação {categoria}"

    if nome_campo not in df_NPS.columns:
        df_NPS[nome_campo] = ""
        campos_notas_NPS.remove(categoria)
        
df_NPS = df_NPS.apply(lambda row: classificacao_NPS(row.name, row, campos_notas_NPS), axis=1)

# # Tratamento de Respostas Nulas
# df_NPS["Resposta"] = df_NPS["Resposta"].fillna("Sem comentário. Coloque como Categoria Não Definida.")
df_NPS = df_NPS.loc[df_NPS["Resposta"].notna()]

display(df_NPS.head(10))

,ASCLI,ASCLI - opção,Ano,Atendimento 0800V,Atendimento 0800V - Opção,CHAVE,Carteira,Ceres,Ceres - opção,Classificação ASCLI,Classificação Atendimento 0800V,Classificação Ceres,Classificação Condições comerciais,Classificação Faturamento,Classificação Frequência de contato EV,Classificação NPS,Classificação Qualidade do atendimento EV,Classificação Qualidade do carro,Classificação Serviços Parceiros,Classificação WebCorp,Cliente,Comentário Sentimento de Compra,Condições comerciais,"Conta para gente, o que motivou sua nota para o processo de comissão?",Código Cliente,Data Pesquisa,Distrital,Divisional,E-mail,Faturamento,Faturamento - Comentário,Faturamento - opção,Frequência de contato - Classificação,Frequência de contato EV,GDS,Há algum serviço ou facilidade que você gostaria que a Localiza oferecesse?,Integração,Intenção de Alugar Carros,Motivação Nota Qualidade do Atendimento EV,Motivação Nota Serviços Parceiros,Motivação Sentimento de Compra,Mês,NPS,O quanto você indicaria sua experiência com nossos processos de pagamento de comissão?,Pesquisa,Qual a sua satisfação com a qualidade do serviço de aluguel de carros prestado pela Localiza?,Qualidade do atendimento EV,Qualidade do carro,Resposta,Sentimento de Falta de Serviços/Facilidade,Serviços Parceiros,Sugestão,Tempo de retorno ASCLI,WebCorp,WebCorp - opção,WebFácil,chave
0,9,NaN,2026,8,NaN,NaN,SSA26A,9,NaN,,Neutro,Promotor,,Promotor,Promotor,Neutro,Promotor,,Neutro,,SOLTEC BRASIL INDUSTRIA COMERCIO E SERVICOS,NaN,7.0,NaN,7431539,2026-08-18,RGNDS,DVCNN,andreia.cortes@soltec.com,9,NaN,NaN,A quantidade de contatos que recebo é SUFICIENTE,9.0,NaN,NaN,NaN,Vou MANTER a quantidade de carros que tenho alugados,clareza e agilidade,demora no envio do orçamento para aprovação do serviço,NaN,8,8.0,NaN,Corporate,NaN,9.0,8.0,"Locações para períodos flexíveis, diversas agencias em todo o Brasil, porém não tem disponibilidade de algumas categorias em determinadas regiões",NaN,8,NaN,NaN,10,NaN,NaN,7431539 | andreia.cortes@soltec.com | 18/08/2026
4,3,NaN,2026,4,NaN,NaN,RON01A,Não aplicável,NaN,,Detrator,NaN,,NaN,Neutro,Detrator,Neutro,,NaN,,AGRO CLUB TECNOLOGICO LTDA,NaN,8.0,NaN,22843644,2026-08-18,RGCOE,DVCNN,durval.carneiro@agroclubtec.com.br,Não aplicável,NaN,NaN,A quantidade de contatos que recebo é SUFICIENTE,7.0,NaN,NaN,NaN,Vou DEVOLVER integralmente os carros que tenho alugados,"O Cara é gente boa, atencioso.",NaN,Custo elevado da locação,8,1.0,NaN,Corporate,NaN,8.0,10.0,"Estava com pagamento em aberto por atrasos no recebimento de algumas comissões e nos programando para efetuar o pagamento quanto antes, 2 parcelas atrasadas e mandaram capangas na casa de um funcionário buscar o carro, imagina a vergonha dele frente a esposa e filhos, sendo que ele não tem culpa de nada, se tivesse combinado deixava o carro lá na própria Localiza, foi muito sem noção.",NaN,Não aplicável,NaN,NaN,Não aplicável,NaN,NaN,22843644 | durval.carneiro@agroclubtec.com.br | 18/08/2026
5,Não aplicável,NaN,2026,9,NaN,NaN,IPT01A,9,NaN,,Promotor,Promotor,,Promotor,Neutro,Neutro,Promotor,,NaN,,PSO - PROJETOS E SOLUCOES OTIMIZADAS LTDA,NaN,8.0,NaN,7226949,2026-08-18,DTMGE,DVSME,wanessa.silva@psoengenharia.com.br,9,NaN,NaN,NaN,8.0,NaN,NaN,NaN,NaN,Atenciosa,NaN,NaN,8,8.0,NaN,Corporate,NaN,9.0,9.0,Ótimo atendimento.,NaN,Não aplicável,NaN,NaN,9,NaN,NaN,7226949 | wanessa.silva@psoengenharia.com.br | 18/08/2026
47,Não aplicável,NaN,2026,7,NaN,NaN,S3I01A,Não aplicável,NaN,,Neutro,NaN,,Detrator,Neutro,Neutro,Neutro,,NaN,,IGREJA PRESBITERIANA DO JARDIM SATE,NaN,7.0,NaN,26709493,2026-08-18,DTIN5,DVINT,alexandreroupa@gmail.com,5,não envio da fatura por email,NaN,A quantidade de contatos que recebo é SUFICIENTE,7.0,NaN,NaN,NaN,Vou MANTER a quantidade de carros que tenho alugados,NaN,NaN,NaN,8,7.0,NaN,Corporate,NaN,7.0,7.0,não envio da fatura por email,NaN,Não aplicável,NaN,NaN,6,NaN,NaN,26709493 | alexandreroupa@gmail.com | 18/08/2026
51,NaN,NaN,2026,NaN,NaN,82026joaopazin@gmail.com,EVI25B,NaN,NaN,,NaN,NaN,,Detrator,Detrator,Det

In [ ]:
import logging
import os
import pandas as pd
import requests
import json
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configura o logging para exibir no console e salvar em arquivo com timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_directory = os.getenv('LOG_DIRECTORY', f"{diretorio_log}")
os.makedirs(log_directory, exist_ok=True)
log_filename = os.path.join(log_directory, f'log_execucao_{timestamp}.log')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()
file_handler = logging.FileHandler(log_filename)
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
logger.addHandler(file_handler)

# Lista para armazenar os logs
log_entries = []

# Função para adicionar logs à lista
def log_to_list(level, message, contrato=None):
    log_entry = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'level': logging.getLevelName(level),
        'message': message,
        'contrato': contrato
    }
    log_entries.append(log_entry)
    logger.log(level, message)

# Log de início de execução
log_to_list(logging.INFO, "Início da execução do script.")    
    
# Verifica se o arquivo CSV existe e carrega os dados existentes
csv_path = os.getenv('CSV_PATH', path_comentarios_existentes)

try:
    with open(csv_path, 'r', encoding='utf-8') as f:
        sample = f.read(1024)
        sniffer = csv.Sniffer()
        dialect = sniffer.sniff(sample)
        sep_detectado = dialect.delimiter
        
    df_existente = pd.read_csv(csv_path, sep=sep_detectado)
    # display(df_existente)
    comentarios_existentes = df_existente["chave"].tolist()
    ultimo_indice = df_existente['Índice'].max() if not df_existente.empty else 0
    log_to_list(logging.INFO, "Arquivo CSV carregado com sucesso.")
    
except FileNotFoundError:
    comentarios_existentes = []
    ultimo_indice = 0
    log_to_list(logging.INFO, "Arquivo CSV não encontrado. Iniciando com dados vazios.")

# dados_respostas = []
def processar_linha(index, row):
    global ultimo_indice
    log_to_list(logging.INFO, f"Processando linha {index}.")
    if row["chave"] not in comentarios_existentes:
        resposta = row.get("Resposta", "") 
        NPS_Indicacao = row.get("Classificação NPS", "")
        pesquisa = row.get("Pesquisa", "")  
        data_pesquisa = row.get("Data Pesquisa", "")  
        codigo_cliente = row.get("Código Cliente", "") 
        cliente = row.get("Cliente", "") 
        email = row.get("E-mail", "") 
        divisional = row.get("Divisional", "") 
        distrital = row.get("Distrital")  
        carteira = row.get("Carteira", "") 
        ano = row.get("Ano", "")  
        mes = row.get("Mes", "")  
        NPS_Atendimento_EV = row.get("Qualidade do atendimento EV","") 
        motivo_Atendimento_EV = row.get("Motivação Nota Qualidade do Atendimento EV","") 
        NPS_Contato_EV = row.get("Frequência de contato EV","") 
        classificacao_Contato_EV = row.get("Frequência de contato - Classificação","")
        NPS_Faturamento = row.get("Faturamento","")
        motivo_Faturamento = row.get("Motivação Nota Faturamento","")
        comentario_Faturamento = row.get("Faturamento - Comentário", "")
        NPS_ServicosParceiros = row.get("Serviços Parceiros","") 
        motivo_ServicosParceiros = row.get("Motivação Nota Serviços Parceiros","")
        NPS_CondicoesComerciais = row.get("Condições comerciais","")
        NPS_Atendimento_Zero_Oitocentos = row.get("Atendimento 0800V","") 
        NPS_Ascli = row.get("ASCLI","") 
        NPS_Ceres = row.get("Ceres","") 
        NPS_Webcorp = row.get("WebCorp","") 
        NPS_QualidadeCarro = row.get("Qualidade do carro","") 
        sentimento_compra = row.get("Intenção de Alugar Carros","") 
        motivo_sentimento_compra = row.get("Motivação Sentimento de Compra","")
        comentario_sentimento_compra = row.get("Comentário Sentimento de Compra","")
        sentimento_de_falta = row.get("Sentimento de Falta de Serviços/Facilidade", "") 
        sugestao = row.get("Sugestão","") 
        chave = row.get("chave", "")

        data = criar_corpo_requisicao(resposta, classificacoes, exemplos_classificacao)
        data_json = json.dumps(data, indent=4)
        
        log_to_list(logging.INFO, "Enviando requisição para o comentário", chave)
        
        tentativas = 0
        sucesso = False
        consumo_total_tokens = 0
        while tentativas < 3 and not sucesso:
            try:
                response = requests.post(url, headers=headers, data=data_json)
                if response.status_code == 200:
                    try:
                        response_data = response.json()
                        log_to_list(logging.INFO, "Requisição bem-sucedida para a resposta", chave)

                        #Controle de Tokens
                        usage = response_data.get("usage", {})  # pode não existir
                        completion_tokens = usage.get("completion_tokens") or usage.get("completionTokens")
                        prompt_tokens     = usage.get("prompt_tokens")     or usage.get("promptTokens")
                        total_tokens      = usage.get("total_tokens")      or usage.get("totalTokens")
                        cost_total = None
                        cost = response_data.get("cost")
                        if isinstance(cost, dict) and isinstance(cost.get("token"), dict):
                            cost_total = cost["token"].get("total")
                            
                        # Controle interno de consumo
                        consumo_total_tokens += total_tokens  # variável global ou externa
                        log_to_list(logging.INFO, f"Tokens usados nesta requisição: {total_tokens} (prompt: {prompt_tokens}, completion: {completion_tokens})", chave)
                        log_to_list(logging.INFO, f"Consumo acumulado até agora: {consumo_total_tokens} tokens", chave)

                        # # Alerta se estiver perto do limite -> pra funcionar precisar criar lógica de armazenamento de tokens entre processamento de requisição
                        # if consumo_total_tokens > limite_tokens * 0.9:
                        #     log_to_list(logging.WARNING, f"Atenção: consumo próximo do limite! {consumo_total_tokens}/{limite}", chave)
                        
                        # Processamento da resposta
                        content_text = response_data["choices"][0]["message"]["content"]
                        content = json.loads(content_text)
                        #print(content_text)
                        content.update({
                            "completionTokens": completion_tokens,
                            "promptTokens": prompt_tokens,
                            "totalTokens": total_tokens,
                            "Custos": cost_total,
                            "Resposta": resposta,
                            "NPS_Indicacao": NPS_Indicacao,
                            "pesquisa": pesquisa,
                            "data_pesquisa": data_pesquisa,
                            "codigo_cliente": codigo_cliente,
                            "cliente": cliente,
                            "e-mail": email,
                            "divisional": divisional,
                            "distrital": distrital,
                            "carteira": carteira,
                            "ano": ano,
                            "mes": mes,
                            "NPS_Atendimento_EV": NPS_Atendimento_EV,
                            "motivo_Atendimento_EV": motivo_Atendimento_EV,
                            "NPS_Contato_EV": NPS_Contato_EV,
                            "classificacao_Contato_EV": classificacao_Contato_EV,
                            "NPS_Faturamento": NPS_Faturamento,
                            "motivo_Faturamento": motivo_Faturamento,
                            "comentario_Faturamento": comentario_Faturamento,
                            "NPS_ServicosParceiros": NPS_ServicosParceiros,
                            "motivo_ServicosParceiros": motivo_ServicosParceiros,
                            "NPS_CondicoesComerciais": NPS_CondicoesComerciais,
                            "NPS_Atendimento_Zero_Oitocentos": NPS_Atendimento_Zero_Oitocentos,
                            "NPS_ASCLI": NPS_Ascli,
                            "NPS_Ceres": NPS_Ceres,
                            "NPS_WebCorp": NPS_Webcorp,
                            "NPS_QualidadeCarro": NPS_QualidadeCarro,
                            "sentimento_compra": sentimento_compra,
                            "motivo_sentimento_compra": motivo_sentimento_compra,
                            "comentario_sentimento_compra": comentario_sentimento_compra,
                            "sentimento_de_falta": sentimento_de_falta,
                            "sugestao": sugestao,
                            "chave": chave,
                            "Índice": ultimo_indice,
                            "Data de Inclusão": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                        })

                        ultimo_indice += 1
                        sucesso = True
                        return content
                    except (KeyError, json.JSONDecodeError) as e:
                        log_to_list(logging.ERROR, f"Erro ao processar resposta para o comentário: {e}", chave)
                        print(content_text)
                else:
                    log_to_list(logging.ERROR, f"Erro na requisição: Status {response.status_code}; {response.text}", chave)
            except (requests.ConnectionError, requests.Timeout, json.JSONDecodeError) as e:
                log_to_list(logging.ERROR, f"Erro na tentativa {tentativas + 1} para o comentário: {e}", chave)
            tentativas += 1
        return None
    else:
        log_to_list(logging.INFO, "Comentário já existente. Pulando.", row["chave"])
    return None

# Utiliza ThreadPoolExecutor para paralelizar as chamadas à API
with ThreadPoolExecutor(max_workers=7) as executor:
    futures = [executor.submit(processar_linha, index, row) for index, row in df_NPS.iterrows()]
    for future in as_completed(futures):
        result = future.result()
        if result:
            dados_respostas.append(result)
            log_to_list(logging.INFO, "Resultado adicionado para o comentário", result['chave'])

log_to_list(logging.INFO, "Finalização da execução do script de requisição.")

# Converte os logs para um DataFrame
df_logs = pd.DataFrame(log_entries)
display(df_logs)

# Verifica se o arquivo de log CSV já existe
log_csv_path = os.path.join(log_directory, 'historico_logs.csv') 
if os.path.exists(log_csv_path):
    df_logs_existente = pd.read_csv(log_csv_path)
    df_logs = pd.concat([df_logs_existente, df_logs], ignore_index=True)

# Salva o DataFrame de logs atualizado no arquivo CSV
df_logs.to_csv(log_csv_path, index=False)

2026-08-24 16:55:00,684 - INFO - Início da execução do script.
2026-08-24 16:55:00,774 - INFO - Arquivo CSV carregado com sucesso.
2026-08-24 16:55:00,914 - INFO - Processando linha 0.
2026-08-24 16:55:00,943 - INFO - Processando linha 4.
2026-08-24 16:55:00,974 - INFO - Enviando requisição para o comentário
2026-08-24 16:55:00,975 - INFO - Processando linha 5.
2026-08-24 16:55:00,984 - INFO - Enviando requisição para o comentário
2026-08-24 16:55:01,010 - INFO - Processando linha 47.
2026-08-24 16:55:01,030 - INFO - Enviando requisição para o comentário
2026-08-24 16:55:01,040 - INFO - Processando linha 51.
2026-08-24 16:55:01,079 - INFO - Processando linha 60.
2026-08-24 16:55:01,123 - INFO - Processando linha 73.
2026-08-24 16:55:01,125 - INFO - Enviando requisição para o comentário
2026-08-24 16:55:01,627 - INFO - Enviando requisição para o comentário
2026-08-24 16:55:01,903 - INFO - Enviando requisição para o comentário
2026-08-24 16:55:01,979 - INFO - Enviando requisição para o c

In [ ]:
# Cria um dataframe com os dados das respostas da API
df_respostas_api = pd.json_normalize(dados_respostas)

# Se o arquivo já existir, concatene os novos dados com os existentes
if comentarios_existentes:
    df_respostas_api = pd.concat([df_existente, df_respostas_api])

# Nova ordem das colunas
nova_ordem = [
    "Índice",
    "Data de Inclusão",
    "Resposta",
    "Classificação",
    "Dúvida",
    "Motivo da Classificação",
    "Sugestão de Nova Categoria",
    "Influencia_Exemplos",
    "Percentual de Assertividade",
    "completionTokens",
    "promptTokens",
    "totalTokens",
    "Custos",
    "NPS_Indicacao",
    "chave",
    "pesquisa",
    "data_pesquisa",
    "codigo_cliente",
    "cliente",
    "e-mail",
    "divisional",
    "distrital",
    "carteira",
    "ano",
    "mes",
    "NPS_Atendimento_EV",
    "NPS_Contato_EV",
    "classificacao_Contato_EV",
    "NPS_Faturamento",
    "motivo_Faturamento",
    "comentario_Faturamento",
    "NPS_ServicosParceiros",
    "motivo_ServicosParceiros",
    "NPS_CondicoesComerciais",
    "NPS_Atendimento_Zero_Oitocentos",
    "NPS_ASCLI",
    "NPS_Ceres",
    "NPS_WebCorp",
    "NPS_QualidadeCarro",
    "sentimento_compra",
    "motivo_sentimento_compra",
    "comentario_sentimento_compra",
    "sentimento_de_falta",
    "sugestao"
]

# Reordenando as colunas
df_respostas_api = df_respostas_api[nova_ordem]

# Ajustando a largura da quinta e sexta colunas (índices 4 e 5)
df_respostas_api_styled = df_respostas_api.style.set_table_styles(
    [
        {
            "selector": "th.col_heading.level0.col2",
            "props": [("min-width", "300px"), ("max-width", "300px")],
        },
        {
            "selector": "td.col2",
            "props": [("min-width", "300px"), ("max-width", "300px")],
        },
        {
            "selector": "th.col_heading.level0.col5",
            "props": [("min-width", "300px"), ("max-width", "300px")],
        },
        {
            "selector": "td.col5",
            "props": [("min-width", "300px"), ("max-width", "300px")],
        },
    ]
)

# Exibindo as 100 primeiras linhas do DataFrame estilizado
df_respostas_api_styled = df_respostas_api.head(20).style


In [ ]:
# Lista dos comentários com problema:

lista_error = df_NPS.loc[~df_NPS["chave"].isin(df_respostas_api["chave"]), "chave"]
lista_error

In [ ]:
# Função para garantir que a coluna "Classificação" seja uma lista
def ensure_list(value):
    if isinstance(value, list):
        return value
    elif isinstance(value, str):
        return value.split(', ')
    else:
        return [value]
 
# Aplicando a função de listagem à coluna "Classificação"
df_respostas_api['Classificação'] = df_respostas_api['Classificação'].apply(ensure_list)

# Explodir as listas em linhas
df_respostas_api = df_respostas_api.explode('Classificação')

# Identifica colunas textuais
text_cols = df_respostas_api.select_dtypes(include=['object']).columns

# Limpa quebras de linha e espaços nas colunas de texto
df_respostas_api[text_cols] = (
    df_respostas_api[text_cols]
        .apply(lambda s: s.astype(str))                           # garante string
        .apply(lambda s: s.str.replace(r'\r?\n', ' ', regex=True))# troca \n por espaço
        .apply(lambda s: s.str.replace(r'\s+', ' ', regex=True))  # colapsa múltiplos espaços
        .apply(lambda s: s.str.strip())                           # remove espaços/linhas em branco no começo/fim
)

#Exportação dos resultados em excel
try:
    df_respostas_api.to_excel(
        path_comentarios_existentes.replace('.csv', '.xlsx'),
        index=False,
        engine='openpyxl'
    )
    print("XLSX exportado com sucesso.")
except Exception as e:
    print(f"Ocorreu um erro ao exportar o arquivo XLSX: {e}")

#Exportação em CSV
try:
    df_respostas_api.to_csv(
        path_comentarios_existentes,
        index=False,
        encoding="utf-8"
    )
    print("CSV exportado com sucesso.")
except Exception as e:
    print(f"Ocorreu um erro ao exportar o arquivo CSV: {e}")

# # Exibe o dataframe
# df_respostas_api
